In [1]:
#OFFLINE INSTALL 
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-index",
    "--find-links", "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages",
    "datasets", "trl", "--ignore-installed"
], stderr=subprocess.DEVNULL)

import datasets, trl
print("datasets:", datasets.__version__, "  trl:", trl.__version__)

datasets: 4.3.0   trl: 0.24.0


In [2]:
# IMPORTS
import os, sys, stat, shutil, gc, zipfile
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


In [3]:
# ── TRITON FIXES ──────────────────────────────────────────────────────────────
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    if os.path.exists(dst_bin): shutil.rmtree(dst_bin)
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

Triton ptxas fix applied.


In [4]:
# ── HYPERPARAMETERS ───────────────────────────────────────────────────────────
SUBSAMPLE_SIZE = 1500   
LORA_RANK      = 32     
LORA_ALPHA     = 32     
LORA_DROPOUT   = 0.0    
MAX_SEQ_LEN    = 1536   
NUM_EPOCHS     = 1
GRAD_ACCUM     = 4
LR             = 1e-4   
OUTPUT_DIR     = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [5]:
# DATA 
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
train_df = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(f"Total samples: {len(train_df)}  →  using {SUBSAMPLE_SIZE}")
train_df   = train_df.sample(n=SUBSAMPLE_SIZE, seed=42)
hf_dataset = Dataset.from_pandas(train_df.to_pandas())

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH, trust_remote_code=True, model_max_length=8192,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


Total samples: 9500  →  using 1500


In [6]:
# FAMILY DETECTION + SYSTEM PROMPTS
SYSTEM_PROMPTS = {
    "bit": (
        "You are an expert in binary operations. Discover the hidden bit "
        "transformation rule (XOR, shift, rotation, bit-flip, nibble-swap) "
        "from the examples. Reason step by step inside <think> tags, then "
        "give your final answer in \\boxed{}."
    ),
    "cipher": (
        "You are an expert cryptographer. Build a letter substitution table "
        "from the examples, verify consistency, then decrypt the text. "
        "Reason inside <think> tags. Final answer in \\boxed{}."
    ),
    "roman": (
        "You are an expert in Roman numerals. "
        "M=1000 CM=900 D=500 CD=400 C=100 XC=90 L=50 XL=40 X=10 IX=9 V=5 IV=4 I=1. "
        "Convert step by step inside <think> tags. Final answer in \\boxed{}."
    ),
    "gravity": (
        "You are a physicist. Use d = 0.5 * g * t² to find the secret g from "
        "the examples (g = 2d/t²), verify it is consistent, then solve for the "
        "target distance. Show work inside <think> tags. Final answer in \\boxed{}."
    ),
    "unit": (
        "You are an expert in unit conversions. Find the conversion factor "
        "(factor = output/input) from the examples, verify it is consistent, "
        "then apply it to the test value. Show work inside <think> tags. "
        "Final answer in \\boxed{}."
    ),
    "symbol": (
        "You are an expert in symbolic pattern recognition. Identify how input "
        "symbols transform to outputs (substitution, position rule, arithmetic "
        "on codes). Apply the rule to the test input. Reason inside <think> "
        "tags. Final answer in \\boxed{}."
    ),
    "other": (
        "You are an expert reasoning assistant. Carefully analyze the pattern "
        "in the examples, show your reasoning inside <think> tags, then give "
        "your final answer in \\boxed{}."
    ),
}

COT_TRACES = {
    "bit": (
        "I'll XOR each input with its output to check for a constant mask.\n"
        "Checking all examples — the mask is consistent across all pairs.\n"
        "Applying the same transformation to the test input."
    ),
    "cipher": (
        "I'll map each encrypted letter to its decrypted counterpart.\n"
        "Building the substitution table from all examples.\n"
        "Applying the table letter by letter to the test text."
    ),
    "roman": (
        "I'll subtract the largest Roman values repeatedly.\n"
        "Thousands → hundreds → tens → ones, using subtractive notation.\n"
        "Combining the symbols gives the result."
    ),
    "gravity": (
        "From each example: g = 2d / t².\n"
        "I calculate g for each pair and confirm it is consistent.\n"
        "Then: d = 0.5 × g × t_test² gives the final distance."
    ),
    "unit": (
        "For each example: factor = output / input.\n"
        "All examples give the same factor — the conversion ratio is confirmed.\n"
        "Applying: result = test_input × factor."
    ),
    "symbol": (
        "Comparing input and output character by character.\n"
        "There is a consistent mapping rule across all examples.\n"
        "Applying the same transformation to the test input."
    ),
    "other": (
        "Examining each input-output pair to identify the pattern.\n"
        "The transformation is consistent across all examples.\n"
        "Applying the discovered rule to the test input."
    ),
}

def detect_family(prompt: str) -> str:
    t = prompt.lower()
    if "bit manipulation rule" in t:                       return "bit"
    if "secret encryption rules" in t:                     return "cipher"
    if "numeral system" in t:                              return "roman"
    if "gravitational constant" in t or "d = 0.5*g*t^2" in t: return "gravity"
    if "unit conversion" in t:                             return "unit"
    if "transformation rules is applied" in t:             return "symbol"
    return "other"

def build_training_text(example):
    prompt  = example["prompt"]
    answer  = str(example["answer"]).strip()
    family  = detect_family(prompt)
    system  = SYSTEM_PROMPTS[family]
    cot     = COT_TRACES[family]

    text = (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}\nPut your final answer inside \\boxed{{}}.<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"<think>\n{cot}\n</think>\n\n"
        f"\\boxed{{{answer}}}<|im_end|>"
    )
    return {"text": text}

hf_dataset = hf_dataset.map(
    build_training_text,
    remove_columns=hf_dataset.column_names
)

# sanity checks
sample = hf_dataset[0]["text"]
assert "<think>" in sample and "</think>" in sample, "Think tags missing!"
assert "\\boxed{" in sample,                          "Boxed answer missing!"
assert "<think></think>" not in sample,               "Empty think block!"
print(f"Format check passed.\n\nSample preview:\n{sample[:600]}\n...")

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Format check passed.

Sample preview:
<|im_start|>system
You are an expert in unit conversions. Find the conversion factor (factor = output/input) from the examples, verify it is consistent, then apply it to the test value. Show work inside <think> tags. Final answer in \boxed{}.<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret unit conversion is applied to measurements. For example:
10.3 m becomes 10.40
12.54 m becomes 12.66
5.85 m becomes 5.91
Now, convert the following measurement: 11.01 m
Put your final answer inside \boxed{}.<|im_end|>
<|im_start|>assistant
<think>
For each example: factor = output / input.
All exam
...


In [7]:
#  MODEL


import unittest.mock as mock


for _mod in [
    'cutlass', 'cutlass.cute',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
    'mamba_ssm.ops.triton.mamba3',
    'mamba_ssm.ops.triton.mamba3.mamba3_mimo_rotary_step',
    'mamba_ssm.modules.mamba3',
]:
    sys.modules[_mod] = mock.MagicMock()


import importlib
for _mod_name in list(sys.modules.keys()):
    if 'mamba_ssm' in _mod_name and 'mamba3' not in _mod_name and 'cute' not in _mod_name:
        try:
            importlib.reload(sys.modules[_mod_name])
        except Exception:
            pass

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
print(f"Model loaded. Vocab size: {len(tokenizer)}")

gc.collect()
torch.cuda.empty_cache()

for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}")

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded. Vocab size: 131072
Patched transformers_modules._1.modeling_nemotron_h


In [8]:
#LORA
lora_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    target_modules = "all-linear",
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


In [9]:
#TRITON COMPILER FIX 
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"


In [10]:
# TRAINING
total_steps  = (len(hf_dataset) // GRAD_ACCUM) * NUM_EPOCHS
warmup_steps = max(5, int(0.05 * total_steps))
print(f"total_steps={total_steps}  warmup_steps={warmup_steps}")

training_args = SFTConfig(
    output_dir                    = OUTPUT_DIR,
    per_device_train_batch_size   = 1,
    gradient_accumulation_steps   = GRAD_ACCUM,
    num_train_epochs              = NUM_EPOCHS,
    learning_rate                 = LR,
    logging_steps                 = 10,
    bf16                          = True,
    max_grad_norm                 = 1.0,
    optim                         = "adamw_torch",
    lr_scheduler_type             = "cosine",
    warmup_steps                  = warmup_steps,
    save_strategy                 = "no",
    report_to                     = "none",
    dataset_text_field            = "text",
    max_length                    = MAX_SEQ_LEN,
    packing                       = False,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": True},
)

trainer = SFTTrainer(
    model            = model,
    train_dataset    = hf_dataset,
    processing_class = tokenizer,
    args             = training_args,
)

print("Starting training...")
trainer.train()


total_steps=375  warmup_steps=18


Adding EOS to train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Starting training...


Step,Training Loss
10,12.249123
20,8.943309
30,4.411306
40,3.383205
50,2.913051
60,2.145503
70,2.712028
80,2.315410
90,2.521239
100,3.083135


TrainOutput(global_step=375, training_loss=2.6845757446289062, metrics={'train_runtime': 4500.5898, 'train_samples_per_second': 0.333, 'train_steps_per_second': 0.083, 'total_flos': 7.0903532744928e+16, 'train_loss': 2.6845757446289062, 'entropy': 0.4870196394622326, 'num_tokens': 368030.0, 'mean_token_accuracy': 0.8668179035186767, 'epoch': 1.0})

In [11]:
# SAVE 
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nAdapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}  ({size/1024:.1f} KB)")


Adapter saved to /kaggle/working/adapter:
  chat_template.jinja  (10.3 KB)
  tokenizer.json  (16677.2 KB)
  tokenizer_config.json  (0.4 KB)
  adapter_model.safetensors  (3454393.7 KB)
  adapter_config.json  (1.1 KB)
  README.md  (5.2 KB)


In [12]:
# DISK CHECK
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nDisk — used: {used/1e9:.1f} GB   free: {free/1e9:.1f} GB")
if free < 2e9:
    print("WARNING: Less than 2GB free — zip may fail!")



Disk — used: 3.6 GB   free: 17.4 GB


In [13]:
# PACKAGE
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f"\nCreated {zip_path}  ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
with zipfile.ZipFile(zip_path, "r") as zf:
    print(f"Contents: {zf.namelist()}")
    assert "adapter_config.json"       in zf.namelist()
    assert "adapter_model.safetensors" in zf.namelist()

print("\nsubmission.zip is ready!")


Created /kaggle/working/submission.zip  (3058.3 MB)
Contents: ['chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'README.md']

submission.zip is ready!
